# Data Preparation / Preprocessing

## Import Pre-processing Libraries

In [24]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

## Load Dataset and Remove Duplicates plus Nulls

In [25]:
# Define file path
file_path = "../data/raw/2025_lar.csv"

# Define Sample Size
sample_size = 1000000

# Load dataset and treat standard missing string tokens as NaN
df = pd.read_csv(file_path, nrows=sample_size, na_values=['', 'Exempt', 'NA', 'na'], low_memory=False)

# Sample dataset
df = df.sample(n=sample_size, random_state=202401617)
print(f"Loaded DataFrame: {df.shape[0]} rows, {df.shape[1]} columns.")

# Remove duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
removed_duplicates = before - len(df)

print(f"Removed {removed_duplicates} duplicate rows | Remaining: {len(df)}")

Loaded DataFrame: 1000000 rows, 99 columns.
Removed 2026 duplicate rows | Remaining: 997974


## Memory Optimization

In [26]:
# Calculate initial memory in MB
initial_memory = df.memory_usage(deep=True).sum() / (1024**2)

# Clean mixed types in total_units by stripping whitespace
df['total_units'] = df['total_units'].astype(str).str.strip()
df['total_units'] = df['total_units'].replace('nan', np.nan)

# Convert string and object columns to category
obj_cols = df.select_dtypes(include=['object', 'string']).columns
df[obj_cols] = df[obj_cols].astype('category')

# Downcast 64-bit floats to 32-bit
float_cols = df.select_dtypes(include='float64').columns
df[float_cols] = df[float_cols].astype('float32')

# Downcast 64-bit integers to 32-bit
int_cols = df.select_dtypes(include='int64').columns
df[int_cols] = df[int_cols].astype('int32')

# Calculate final memory in MB
final_memory = df.memory_usage(deep=True).sum() / (1024**2)
reduction = (1 - final_memory / initial_memory) * 100

print(f"Memory Usage: {initial_memory:.2f} MB -> {final_memory:.2f} MB ({reduction:.1f}% reduction)")

Memory Usage: 1413.15 MB -> 337.96 MB (76.1% reduction)


## Drop Unneeded, Leaky, Identifier, and Redundant Features

In [27]:
# Post-decision fields that leak the target variable
post_cols = [
    'denial_reason_1', 'denial_reason_2', 'denial_reason_3', 'denial_reason_4',
    'purchaser_type', 'interest_rate', 'rate_spread', 'total_loan_costs',
    'total_points_and_fees', 'origination_charges', 'discount_points',
    'lender_credits', 'intro_rate_period', 'prepayment_penalty_term', 'hoepa_status'
]

# Identifiers, constant year, and high-cardinality IDs (including lei)
identifier_cols = [
    'activity_year',
    'lei',
    'derived_msa_md',
    'county_code',
    'census_tract'
]

# Redundant derived or duplicate columns
redundant_cols = [
    'derived_loan_product_type',
    'applicant_age_above_62',
    'co_applicant_age_above_62'
]

# Protected demographic attributes
protected_cols = [
    'derived_ethnicity', 'derived_race', 'derived_sex',
    'applicant_ethnicity_1', 'applicant_ethnicity_2', 'applicant_ethnicity_3',
    'applicant_ethnicity_4', 'applicant_ethnicity_5', 'co_applicant_ethnicity_1',
    'co_applicant_ethnicity_2', 'co_applicant_ethnicity_3', 'co_applicant_ethnicity_4',
    'co_applicant_ethnicity_5', 'applicant_ethnicity_observed', 'co_applicant_ethnicity_observed',
    'applicant_race_1', 'applicant_race_2', 'applicant_race_3', 'applicant_race_4',
    'applicant_race_5', 'co_applicant_race_1', 'co_applicant_race_2', 'co_applicant_race_3',
    'co_applicant_race_4', 'co_applicant_race_5', 'applicant_race_observed',
    'co_applicant_race_observed', 'applicant_sex', 'co_applicant_sex',
    'applicant_sex_observed', 'co_applicant_sex_observed'
]

# Extra automated underwriting system slots
aus_extra_cols = ['aus_2', 'aus_3', 'aus_4', 'aus_5']

# Drop all defined unneeded columns
drop_cols = post_cols + identifier_cols + redundant_cols + protected_cols + aus_extra_cols
df = df.drop(columns=drop_cols, errors='ignore')

print(f"Dropped {len(drop_cols)} unneeded/leaky columns | Remaining columns: {df.shape[1]}")

Dropped 58 unneeded/leaky columns | Remaining columns: 41


## Drop Sparse and Constant Columns

In [28]:
# Identify columns with >60% missing values
sparse_cols = []
for col in df.columns:
    if df[col].isna().mean() > 0.60:
        sparse_cols.append(col)

# Identify columns with no unique columns
constant_cols = []
for col in df.columns:
    if df[col].nunique(dropna=True) <= 1:
        constant_cols.append(col)

# Add both lists and drop the columns
cols_to_remove = sparse_cols + constant_cols
df = df.drop(columns=cols_to_remove, errors='ignore')

print(f"Sparse columns dropped (>60% missing): {sparse_cols}")
print(f"Constant columns dropped: {constant_cols}")
print(f"Remaining columns: {df.shape[1]}")

Sparse columns dropped (>60% missing): ['multifamily_affordable_units']
Constant columns dropped: []
Remaining columns: 40


## Dataset Cleaning

In [29]:
# Rescale monetary values to thousands (1K$) to align with income scale
df['loan_amount'] = (df['loan_amount'] / 1000).astype('float32')
df['property_value'] = (df['property_value'] / 1000).astype('float32')

# Replace impossible or sentinel numeric values with NaN across the dataset
df['loan_amount'] = df['loan_amount'].where(df['loan_amount'] > 0)
df['income'] = df['income'].where(df['income'] > 0)
df['property_value'] = df['property_value'].where(df['property_value'] > 0)
df['combined_loan_to_value_ratio'] = df['combined_loan_to_value_ratio'].where(
    df['combined_loan_to_value_ratio'].between(1, 150)
)
df['tract_median_age_of_housing_units'] = df['tract_median_age_of_housing_units'].where(
    df['tract_median_age_of_housing_units'] > 0
)
df['tract_population'] = df['tract_population'].where(df['tract_population'] > 0)
df['ffiec_msa_md_median_family_income'] = df['ffiec_msa_md_median_family_income'].where(
    df['ffiec_msa_md_median_family_income'] > 0
)

print("Replaced invalid sentinel values with NaN and aligned monetary units.")

Replaced invalid sentinel values with NaN and aligned monetary units.


## Feature Engineering

In [30]:
# Create binary co-applicant presence indicator
df['has_coapplicant'] = (df['co_applicant_age'].astype(str) != '9999').astype('int8')

# Calculate loan-to-income affordability ratio
df['loan_to_income'] = (df['loan_amount'] / df['income']).astype('float32')

print("Engineered features: 'has_coapplicant' and 'loan_to_income'.")

Engineered features: 'has_coapplicant' and 'loan_to_income'.


## Separate Features and Multi-Class Target

In [31]:
target_col = 'action_taken'

# Drop missing target rows
df = df.dropna(subset=[target_col]).reset_index(drop=True)

# Separate features (X) and target (y)
X = df.drop(columns=[target_col], errors='ignore')
y = df[target_col]

# Encode target classes (1-8 -> 0-7)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print(f"Feature Matrix Shape: {X.shape}")
print(f"Target Classes ({len(label_encoder.classes_)}): {label_encoder.classes_}")

Feature Matrix Shape: (997974, 41)
Target Classes (8): [1 2 3 4 5 6 7 8]


## Train-Test Split

In [32]:
# Stratified train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=202401617,
    stratify=y_encoded
)

print(f"Training Set Shape: {X_train.shape}")
print(f"Testing Set Shape:  {X_test.shape}")

Training Set Shape: (798379, 41)
Testing Set Shape:  (199595, 41)


## Outlier Handling

In [33]:
# Numerical columns to cap extreme values
cap_cols = [
    'loan_amount',
    'income',
    'property_value',
    'loan_to_income',
    'loan_term',
    'tract_population',
    'ffiec_msa_md_median_family_income'
]

# Mask 1st and 99th percentiles from X_train only
lower_caps = X_train[cap_cols].quantile(0.01)
upper_caps = X_train[cap_cols].quantile(0.99)

# Clip extreme values on both train and test splits
X_train[cap_cols] = X_train[cap_cols].clip(lower=lower_caps, upper=upper_caps, axis=1)
X_test[cap_cols] = X_test[cap_cols].clip(lower=lower_caps, upper=upper_caps, axis=1)

# Apply log1p transform to right-skewed financial features
log_cols = ['loan_amount', 'income', 'property_value', 'loan_to_income']
X_train[log_cols] = np.log1p(X_train[log_cols])
X_test[log_cols] = np.log1p(X_test[log_cols])

## Define Preprocessing Pipeline

In [34]:
# Define numerical features in a list
num_cols = [
    'loan_amount',
    'property_value',
    'income',
    'combined_loan_to_value_ratio',
    'loan_term',
    'tract_population',
    'tract_minority_population_percent',
    'ffiec_msa_md_median_family_income',
    'tract_to_msa_income_percentage',
    'tract_owner_occupied_units',
    'tract_one_to_four_family_homes',
    'tract_median_age_of_housing_units',
    'loan_to_income'
]

# Define categorical features in a list
cat_cols = [col for col in X_train.columns if col not in num_cols]

# Impute categorical missing values with explicit string label
for col in cat_cols:
    X_train[col] = X_train[col].astype(object).fillna('Missing').astype(str)
    X_test[col] = X_test[col].astype(object).fillna('Missing').astype(str)

# Numerical pipeline: Median imputation with missing indicators (no scaler)
num_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True))
])

# One-Hot Encoding rare categories (<1%) into 'infrequent'
cat_pipe = OneHotEncoder(
    handle_unknown='infrequent_if_exist',
    min_frequency=0.01,
    sparse_output=False,
    dtype=np.float32
)

# Combine pipelines
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipe, num_cols),
        ('cat', cat_pipe, cat_cols)
    ],
    verbose_feature_names_out=False
)

# Output DataFrames directly onto X_train and X_test
preprocessor.set_output(transform="pandas")

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

print(f"Train Shape: {X_train.shape} | Test Shape: {X_test.shape}")
print(f"Remaining NaNs -> Train: {X_train.isna().sum().sum()} | Test: {X_test.isna().sum().sum()}")

Train Shape: (798379, 185) | Test Shape: (199595, 185)
Remaining NaNs -> Train: 0 | Test: 0


## Save Datasets as CSV

In [35]:
# Reattach encoded target variable
train_final = X_train.copy()
train_final['action_taken'] = y_train

test_final = X_test.copy()
test_final['action_taken'] = y_test

# Save directly to target CSV locations
train_csv_path = "../data/processed/hmda_2025_train_preprocessed.csv"
test_csv_path = "../data/processed/hmda_2025_test_preprocessed.csv"

train_final.to_csv(train_csv_path, index=False)
test_final.to_csv(test_csv_path, index=False)

print(f"Saved preprocessed training data to: {train_csv_path}")
print(f"Saved preprocessed testing data to:  {test_csv_path}")

Saved preprocessed training data to: ../data/processed/hmda_2025_train_preprocessed.csv
Saved preprocessed testing data to:  ../data/processed/hmda_2025_test_preprocessed.csv
